# 06 — Decision Engine
**Goal:** Combine outputs from all 4 models into one overall risk assessment per link.

Load the datasets and check how many IPs they share:

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df_sanity = pd.read_csv(PROCESSED_DIR / "sanity_features.csv")
df_rxlev = pd.read_csv(PROCESSED_DIR / "rxlev_features.csv")
df_fh_rsl = pd.read_csv(PROCESSED_DIR / "fh_rsl_features.csv")

sanity_ips = set(df_sanity["IP"].unique())
rxlev_ips = set(df_rxlev["IP"].unique())
fh_rsl_ips = set(df_fh_rsl["IP"].unique())

print("unique IPs — sanity:", len(sanity_ips))
print("unique IPs — rxlev:", len(rxlev_ips))
print("unique IPs — fh_rsl:", len(fh_rsl_ips))

print()
print("sanity ∩ rxlev:", len(sanity_ips & rxlev_ips))
print("sanity ∩ fh_rsl:", len(sanity_ips & fh_rsl_ips))
print("rxlev ∩ fh_rsl:", len(rxlev_ips & fh_rsl_ips))
print("all three:", len(sanity_ips & rxlev_ips & fh_rsl_ips))

unique IPs — sanity: 3311
unique IPs — rxlev: 3307
unique IPs — fh_rsl: 1292

sanity ∩ rxlev: 3307
sanity ∩ fh_rsl: 966
rxlev ∩ fh_rsl: 963
all three: 963


Build the combined per-IP view

In [2]:
# get one row per IP in each dataset — for sanity/fh_rsl, keep the "worst" status if an IP has multiple rows
severity_order_sanity = {"OK": 0, "PREVENTIVE": 1, "CURATIVE": 2}
severity_order_fh = {"Lien_OK": 0, "Lien_dépointé_(<10)": 1, "Lien_dépointé_(>10)": 2}

df_sanity["_severity"] = df_sanity["Sanity"].map(severity_order_sanity)
sanity_per_ip = df_sanity.loc[df_sanity.groupby("IP")["_severity"].idxmax()]

df_fh_rsl["_severity"] = df_fh_rsl["Status"].map(severity_order_fh)
fh_rsl_per_ip = df_fh_rsl.loc[df_fh_rsl.groupby("IP")["_severity"].idxmax()]

# rxlev has no label — just aggregate rsl_range by mean per IP as a simple summary
rxlev_per_ip = df_rxlev.groupby("IP").agg({"rsl_range": "mean", "Min RSL": "mean"}).reset_index()

print("sanity_per_ip:", sanity_per_ip.shape)
print("fh_rsl_per_ip:", fh_rsl_per_ip.shape)
print("rxlev_per_ip:", rxlev_per_ip.shape)

# join all three on IP
combined = sanity_per_ip[["IP", "Sanity"]].merge(
    fh_rsl_per_ip[["IP", "Status"]], on="IP", how="inner"
).merge(
    rxlev_per_ip, on="IP", how="inner"
)

print("\ncombined shape:", combined.shape)
display(combined.head(10))

sanity_per_ip: (3311, 19)
fh_rsl_per_ip: (1292, 10)
rxlev_per_ip: (3307, 3)

combined shape: (963, 5)


,IP,Sanity,Status,rsl_range,Min RSL
0,172.18.10.11,PREVENTIVE,Lien_dépointé_(<10),NaN,-39.8
1,172.18.10.18,OK,Lien_OK,9.9,-38.4
2,172.18.10.2,CURATIVE,Lien_OK,NaN,NaN
3,172.18.10.3,CURATIVE,Lien_OK,34.3,-67.2
4,172.18.10.5,OK,Lien_OK,10.3,-39.7
5,172.18.10.7,PREVENTIVE,Lien_dépointé_(<10),NaN,-42.0
6,172.18.10.8,PREVENTIVE,Lien_dépointé_(>10),21.3,-54.8
7,172.18.10.9,PREVENTIVE,Lien_OK,6.0,-97.4
8,172.18.100.10,PREVENTIVE,Lien_dépointé_(<10),NaN,-31.6
9,172.18.100.13,PREVENTIVE,Lien_OK,32.1,-63.8


Build the actual risk-scoring logic

In [3]:
def compute_overall_risk(row):
    """Combine Sanity, fh_rsl Status, and rxlev signal into one risk level + reasons."""
    reasons = []
    risk_points = 0

    # Sanity contribution
    if row["Sanity"] == "CURATIVE":
        risk_points += 2
        reasons.append("Sanity classifier flags CURATIVE (urgent repair)")
    elif row["Sanity"] == "PREVENTIVE":
        risk_points += 1
        reasons.append("Sanity classifier flags PREVENTIVE (scheduled maintenance)")

    # fh_rsl contribution
    if row["Status"] == "Lien_dépointé_(>10)":
        risk_points += 2
        reasons.append("FH RSL classifier flags severe misalignment (>10)")
    elif row["Status"] == "Lien_dépointé_(<10)":
        risk_points += 1
        reasons.append("FH RSL classifier flags mild misalignment (<10)")

    # rxlev contribution — only if data exists
    if pd.notna(row["Min RSL"]) and row["Min RSL"] < -95:
        risk_points += 1
        reasons.append(f"RXLev shows very weak minimum signal ({row['Min RSL']} dBm)")

    # model disagreement flag
    sanity_bad = row["Sanity"] in ("CURATIVE", "PREVENTIVE")
    fh_bad = row["Status"] != "Lien_OK"
    if sanity_bad != fh_bad:
        reasons.append("Note: Sanity and FH RSL classifiers disagree on this link")

    # overall risk level
    if risk_points >= 3:
        risk_level = "HIGH"
    elif risk_points >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return pd.Series({"risk_level": risk_level, "risk_points": risk_points, "reasons": "; ".join(reasons)})

risk_results = combined.apply(compute_overall_risk, axis=1)
combined_with_risk = pd.concat([combined, risk_results], axis=1)

print(combined_with_risk["risk_level"].value_counts())
print()
display(combined_with_risk.sort_values("risk_points", ascending=False).head(10))

risk_level
MEDIUM    596
HIGH      224
LOW       143
Name: count, dtype: int64



,IP,Sanity,Status,rsl_range,Min RSL,risk_level,risk_points,reasons
503,172.18.92.25,CURATIVE,Lien_dépointé_(>10),69.15,-96.80,HIGH,5,Sanity classifier flags CURATIVE (urgent repai...
927,172.19.78.13,CURATIVE,Lien_dépointé_(>10),NaN,-97.20,HIGH,5,Sanity classifier flags CURATIVE (urgent repai...
926,172.19.78.10,CURATIVE,Lien_dépointé_(>10),NaN,-97.90,HIGH,5,Sanity classifier flags CURATIVE (urgent repai...
701,172.19.198.215,CURATIVE,Lien_dépointé_(>10),66.98,-96.10,HIGH,5,Sanity classifier flags CURATIVE (urgent repai...
656,172.19.180.16,CURATIVE,Lien_dépointé_(>10),64.45,-97.45,HIGH,5,Sanity classifier flags CURATIVE (urgent repai...
501,172.18.90.1,CURATIVE,Lien_dépointé_(<10),70.70,-97.80,HIGH,4,Sanity classifier flags CURATIVE (urgent repai...
67,172.18.128.21,CURATIVE,Lien_dépointé_(>10),NaN,NaN,HIGH,4,Sanity classifier flags CURATIVE (urgent repai...
523,172.19.104.13,CURATIVE,Lien_dépointé_(>10),NaN,NaN,HIGH,4,Sanity classifier flags CURATIVE (urgent repai...
860,172.19.42.6,CURATIVE,Lien_dépointé_(<10),58.40,-95.70,HIGH,4,Sanity classifier flags CURATIVE (urgent repai...
414,172.18.66.2,CURATIVE,Lien_dépointé_(<10),60.40,-98.00,HIGH,4,Sanity classifier flags CURATIVE (urgent repai...


Finish the reasons field and save the decision engine as reusable code

In [4]:
print(combined_with_risk.loc[503, "reasons"])

Sanity classifier flags CURATIVE (urgent repair); FH RSL classifier flags severe misalignment (>10); RXLev shows very weak minimum signal (-96.8 dBm)


In [7]:
output_path = Path("../data/processed/network_risk_assessment.csv")
combined_with_risk.to_csv(output_path, index=False)
print("saved:", output_path.resolve())

saved: C:\OrangeCopilot\projet_pfe-main (1)\projet_pfe-main\main\FH\AI\data\processed\network_risk_assessment.csv


Step — Investigate possible matching keys

In [8]:
# look at what identifiers actually look like across datasets
print("--- fh_rsl Name examples ---")
print(df_fh_rsl["Name"].head(10).tolist())

print("\n--- rxlev Name examples ---")
print(df_rxlev["Name"].head(10).tolist())

print("\n--- rtwp Cell_Name examples ---")
df_rtwp_full = pd.read_csv(PROCESSED_DIR / "rtwp_features.csv")
print(df_rtwp_full["Cell_Name"].head(10).tolist())

print("\n--- atoll_links link_id examples ---")
df_atoll_full = pd.read_csv(PROCESSED_DIR / "atoll_links_features.csv")
print(df_atoll_full["link_id"].head(10).tolist())

--- fh_rsl Name examples ---
['B2B_TTDE_TUN_0058', 'B2B_ATB_KELIBIA_NAB_0061', 'B2B_ATB_CHOTRANA_ARI_0092', 'B2B_ISB_SFA_0001', 'B2B_BT_MORNEG_BAR_0046', 'B2B_MONOPRIX_GABES_GAB_0027', 'B2B_MALTEX_MON_0017', 'B2B_SOTRAMI_KEF_0022', 'B2B_ATS_TUN_0052', 'B2B_SAGEM_BAR_0046']

--- rxlev Name examples ---
['SFA_0112_SBO_0015_SFA_0330', 'ARI_0166_JUMIA_SERV_CTF_Charguia', 'JEN_0038_JEN_0003', 'SFA_0104_SFA_105', 'TUN_0092_FRANCHISE_MOUROUJ1', 'SBO_0039_SBO_0043_LTE', 'KAI_0101_KAI_0052', 'MAH_0049_MAH_0090', 'SBO_0039_SBO_0043_LTE', 'TOZ_0010_ATB_TOZEUR']

--- rtwp Cell_Name examples ---
['ARI_0017_C04_309_w3', 'ARI_0025_C03_309_w4', 'ARI_0029_C01_309_w3', 'ARI_0033_C09_309_w3', 'ARI_0034_C02_309_v8', 'ARI_0042_C02_309_v7', 'ARI_0042_C02_309_v8', 'ARI_0051_C01_409_w2', 'ARI_0066_C07_110_w1', 'ARI_0076_C02_110_v9']

--- atoll_links link_id examples ---
['2AP_CONSEIL_Tunis - ARI_0012_C01', 'ABARAKA_Palmarium - SFA_0040_C01', 'ABATTOIR_CHAHIA_Zarzis - MED_0025_C03', 'ABT_TUN_0009', 'ADP_CHARGU

Extract site codes and test the overlap

In [9]:
import re

def extract_site_code(text):
    """Extract a site code like 'TUN_0058' or 'ARI_0092' from a longer identifier."""
    if pd.isna(text):
        return None
    match = re.search(r"[A-Z]{2,4}_\d{3,5}", str(text))
    return match.group(0) if match else None

df_fh_rsl["site_code"] = df_fh_rsl["Name"].apply(extract_site_code)
df_rxlev["site_code"] = df_rxlev["Name"].apply(extract_site_code)
df_rtwp_full["site_code"] = df_rtwp_full["Cell_Name"].apply(extract_site_code)
df_atoll_full["site_code"] = df_atoll_full["link_id"].apply(extract_site_code)
df_sanity["site_code"] = None  # sanity has no Name column — check this below

print("sanity columns:", df_sanity.columns.tolist())

for name, df in [("fh_rsl", df_fh_rsl), ("rxlev", df_rxlev), ("rtwp", df_rtwp_full), ("atoll", df_atoll_full)]:
    n_extracted = df["site_code"].notna().sum()
    print(f"{name}: {n_extracted}/{len(df)} site codes extracted, {df['site_code'].nunique()} unique")

# overlap check
fh_codes = set(df_fh_rsl["site_code"].dropna())
rxlev_codes = set(df_rxlev["site_code"].dropna())
rtwp_codes = set(df_rtwp_full["site_code"].dropna())
atoll_codes = set(df_atoll_full["site_code"].dropna())

print("\nfh_rsl ∩ rtwp:", len(fh_codes & rtwp_codes))
print("fh_rsl ∩ atoll:", len(fh_codes & atoll_codes))
print("rxlev ∩ rtwp:", len(rxlev_codes & rtwp_codes))
print("rxlev ∩ atoll:", len(rxlev_codes & atoll_codes))

sanity columns: ['IP', 'Slot', 'Sanity', 'Mod (Ref)', 'Mod (Min)', 'UAS', 'SEP', 'SES', 'ES', 'BBE', 'OFS', 'RSL (Min)', 'RSL (Max)', 'RSL (Avg)', 'RSL (Min)_was_sentinel', 'RSL (Max)_was_sentinel', 'RSL (Avg)_was_sentinel', 'rsl_range', '_severity', 'site_code']
fh_rsl: 1268/1292 site codes extracted, 313 unique
rxlev: 5959/6016 site codes extracted, 1676 unique
rtwp: 295/295 site codes extracted, 180 unique
atoll: 3785/3795 site codes extracted, 1587 unique

fh_rsl ∩ rtwp: 24
fh_rsl ∩ atoll: 297
rxlev ∩ rtwp: 89
rxlev ∩ atoll: 1366
